In [ ]:
import datahub.emitter.concepts as concepts
from datahub.emitter.mcp import MetadataChangeProposalWrapper
from datahub.emitter.rest_emitter import DatahubRestEmitter
from datahub.metadata.schema_classes import (
    DatasetPropertiesClass,
    SchemaMetadataClass,
    EditableSchemaMetadataClass,
    EditableSchemaFieldInfoClass,
    MySqlIdentifierClass,
    StringTypeClass,
    NumberTypeClass,
    OwnershipClass,
    OwnerClass,
    OwnershipTypeClass,
)

# Verbindung zum lokalen DataHub GMS (Standard-Port 8080)
emitter = DatahubRestEmitter("http://localhost:8080")

dataset_urn = "urn:li:dataset:(urn:li:dataPlatform:postgres,raw-orders,PROD)"

# 1. Metadaten: Basis-Eigenschaften und Spalten-Schema
schema_metadata = SchemaMetadataClass(
    schemaName="raw-orders",
    platform="urn:li:dataPlatform:postgres",
    version=0,
    hash="",
    platformSchema=MySqlIdentifierClass(table="raw-orders"),
    fields=[
        concepts.Field(
            fieldPath="order_id",
            type=NumberTypeClass(),
            description="Eindeutige ID der Bestellung (Primary Key)."
        ),
        concepts.Field(
            fieldPath="customer_email",
            type=StringTypeClass(),
            description="Email des Kunden. Enthält PII Daten!"
        ),
        concepts.Field(
            fieldPath="total_amount",
            type=NumberTypeClass(),
            description="Bruttobetrag der Bestellung in EUR."
        )
    ]
)

# 2. Dataset Properties (Beschreibung der Tabelle)
dataset_properties = DatasetPropertiesClass(
    description="Diese Tabelle enthält alle rohen Bestelldaten aus dem Webshop-Backend.",
    externalUrl="https://wiki.deinefirma.de/daten/orders"
)

# 3. Ownership setzen (Beispiel: Data Steward)
ownership = OwnershipClass(
    owners=[
        OwnerClass(
            owner="urn:li:corpuser:admin", # Der Standard-Admin
            type=OwnershipTypeClass.DATA_STEWARD
        )
    ]
)

# Die Metadaten an DataHub senden
def emit_metadata(aspect_name, aspect_value):
    mcp = MetadataChangeProposalWrapper(
        entityType="dataset",
        changeType="UPSERT",
        entityUrn=dataset_urn,
        aspectName=aspect_name,
        aspect=aspect_value,
    )
    emitter.emit(mcp)

print(f"Sende Metadaten für {dataset_urn}...")
emit_metadata("schemaMetadata", schema_metadata)
emit_metadata("datasetProperties", dataset_properties)
emit_metadata("ownership", ownership)
print("Erfolgreich übertragen! Prüfe nun dein DataHub Frontend.")